import IBM AMLSim

In [1]:
from loader.dataset_factory import DatasetFactory

# Inizializza il loader specificando il nome canonico del dataset
loader = DatasetFactory.get_loader("ibm_amlsim")

# Carica i dati in memoria (file delle transazioni e, se presente, degli account)
loader.load()

# Stampa un riepilogo conciso del dataset caricato per verificare che sia tutto ok
loader.summary()

Loading IBM AMLSim transactions from: c:\Users\marti\Desktop\Magistrale\Tesi v2\Code\loader\..\data\ibm_amlsim\HI-Small_Trans.csv
  Loaded 5,078,345 rows.
Loading IBM AMLSim accounts from: c:\Users\marti\Desktop\Magistrale\Tesi v2\Code\loader\..\data\ibm_amlsim\HI-Small_accounts.csv
  Loaded 518,581 account records.
  Dataset : IBMAMLSimLoader
  Rows    : 5,078,345
  Columns : 11
  Fraud   : 5,177  (0.1019 %)



In [2]:
loader.print_features()


Column               Type         Description
--------------------------------------------------------------------------------
Timestamp            object       Date and time of the transaction
From Bank            int64        Numeric identifier of the originating bank  (e.g. [1, 10, 12, 3208, 3209])
Account              object       Hexadecimal ID of the sender account (node)
To Bank              int64        Numeric identifier of the receiving bank  (e.g. [1, 10, 12, 2439, 3209])
Account.1            object       Hexadecimal ID of the receiver account (node)
Amount Received      float64      Transaction amount in the receiving currency
Receiving Currency   object       Currency received by the destination account  (e.g. ['US Dollar', 'Bitcoin', 'Euro', 'Australian Dollar', 'Yuan', 'Rupee'])
Amount Paid          float64      Transaction amount in the paying currency
Payment Currency     object       Currency used by the sender  (e.g. ['US Dollar', 'Bitcoin', 'Euro', 'Australian Doll

Data preparation

In [3]:
from data_preparation import DataPreparation

# Otteniamo il dataframe raw delle transazioni dal nostro loader
transactions_df = loader.get_transactions()

# Inizializziamo la classe scegliendo il robust scaler
data_prep = DataPreparation(scaler_type='robust')

# 1. Calcoliamo le feature standardizzate per le transazioni (gli archi E)
edges_features_df = data_prep.fit_transform_edges(transactions_df)

# 2. Aggreghiamo le transazioni per ricavare l'embedding iniziale dei nodi (V)
nodes_features_df = data_prep.get_node_features(edges_features_df)

print(f"Dimensione feature archi: {edges_features_df.shape}")
print(f"Dimensione feature nodi: {nodes_features_df.shape}")

# Stampa bilanciamento classi (Nodi e Archi)
print("\n--- Bilanciamento Dataset ---")
num_fraud_nodes = (nodes_features_df["Is Laundering"] > 0).sum()
num_legit_nodes = len(nodes_features_df) - num_fraud_nodes
print(f"Nodi fraudolenti: {num_fraud_nodes} ({num_fraud_nodes/len(nodes_features_df)*100:.2f}%)")
print(f"Nodi leciti:      {num_legit_nodes} ({num_legit_nodes/len(nodes_features_df)*100:.2f}%)")

num_fraud_edges = (edges_features_df["Is Laundering"] > 0).sum()
num_legit_edges = len(edges_features_df) - num_fraud_edges
print(f"Archi fraudolenti: {num_fraud_edges} ({num_fraud_edges/len(edges_features_df)*100:.2f}%)")
print(f"Archi leciti:      {num_legit_edges} ({num_legit_edges/len(edges_features_df)*100:.2f}%)")


Extracting time features (Unix time, Cyclic Hour, Day of Week)...
Applying edge transformation (Amounts: robust, Time: standard)...
Aggregating node features and extracting ground truth labels...
Dimensione feature archi: (5078345, 52)
Dimensione feature nodi: (515080, 49)

--- Bilanciamento Dataset ---
Nodi fraudolenti: 6357 (1.23%)
Nodi leciti:      508723 (98.77%)
Archi fraudolenti: 5177 (0.10%)
Archi leciti:      5073168 (99.90%)



### GAGNN Model Setup & Hyperparameters

Below are the key hyperparameters for the GAGNN implementation. All values are defined in the code cell below.

**Model Architecture:**
- `node_in_dim`: Dimension of input node features (inferred from data).
- `edge_feat_dim`: Dimension of edge features (inferred from data).
- `hidden_dim`: Hidden dimension for GAT layers (from paper's GAT#1 units).
- `out_dim`: Output dimension of the community-centric encoder.
- `heads`: Number of attention heads $k$ (as per paper).
- `beta`: Trade-off parameter for eMRF similarity.

**Loss Optimization ($\mathcal{L} = c_1 \mathcal{L}_{group} + c_2 \mathcal{L}_{node} + c_3 \mathcal{L}_{trans}$):**
- `c1`: Weight for the group-level loss term.
- `c2`: Weight for the node-level loss term.
- `c3`: Weight for the transaction-level loss term.

**Training:**
- `learning_rate`: Learning rate for the Adam optimizer (from paper).
- `epochs`: Number of full passes over the training batches.

**Mini-batch Sampling (NeighborLoader):**
- `batch_size`: Number of seed nodes processed per mini-batch.
- `num_neighbors`: List of neighbors sampled per hop for each GAT layer. Length must match the number of GAT layers (2).


In [4]:
import torch
import numpy as np
import pandas as pd

print("--- Downsampling Strategy (1:1 Ratio) ---")
# 1. Identify Fraudulent and Legitimate Nodes
# A node is fraudulent if it was involved in at least one ML transaction
fraud_nodes = nodes_features_df[nodes_features_df["Is Laundering"] > 0].index
legit_nodes = nodes_features_df[nodes_features_df["Is Laundering"] == 0].index

print(f"Total Fraudulent Nodes: {len(fraud_nodes)}")
print(f"Total Legitimate Nodes (before sampling): {len(legit_nodes)}")

# 2. Sample Legitimate Nodes (1:1 ratio)
sampled_legit_nodes = pd.Series(legit_nodes).sample(n=len(fraud_nodes), random_state=42).values

# 3. Combine to form the final set of sampled users
sampled_users = set(fraud_nodes).union(set(sampled_legit_nodes))
print(f"Total Nodes after downsampling: {len(sampled_users)}")

# 4. Filter nodes_features_df and edges_features_df efficiently
nodes_features_df = nodes_features_df.loc[list(sampled_users)]

# Filter transactions: keep only those where BOTH sender and receiver are in sampled_users
edges_features_df = edges_features_df[
    edges_features_df['Account'].isin(sampled_users) & 
    edges_features_df['Account.1'].isin(sampled_users)
]

print(f"Total Edges after downsampling: {len(edges_features_df)}")

# 5. Map string Account IDs to integers (0 to N-1) for PyTorch Geometric
unique_nodes = nodes_features_df.index.unique()
node_mapping = pd.Series(index=unique_nodes, data=np.arange(len(unique_nodes)))

# 6. Extract edge_index
src = edges_features_df['Account'].map(node_mapping).values
dst = edges_features_df['Account.1'].map(node_mapping).values
edge_index = torch.tensor(np.vstack((src, dst)), dtype=torch.long)

# 7. Extract edge features and transaction labels
edge_features_cols = [c for c in edges_features_df.columns if c not in ['Account', 'Account.1', 'Is Laundering', 'Timestamp']]
edge_attr = torch.tensor(edges_features_df[edge_features_cols].values, dtype=torch.float)
y_trans = torch.tensor(edges_features_df['Is Laundering'].values, dtype=torch.float).unsqueeze(1)

# 8. Extract node features and node labels
node_features_cols = [c for c in nodes_features_df.columns if c != 'Is Laundering']
x = torch.tensor(nodes_features_df[node_features_cols].values, dtype=torch.float)

y_node = torch.tensor(nodes_features_df['Is Laundering'].values, dtype=torch.float)

print(f"Node features shape: {x.shape}")
print(f"Edge index shape: {edge_index.shape}")
print(f"Edge features shape: {edge_attr.shape}")


--- Downsampling Strategy (1:1 Ratio) ---
Total Fraudulent Nodes: 6357
Total Legitimate Nodes (before sampling): 508723
Total Nodes after downsampling: 12714
Total Edges after downsampling: 52180
Node features shape: torch.Size([12714, 48])
Edge index shape: torch.Size([2, 52180])
Edge features shape: torch.Size([52180, 48])


In [5]:
# =============================================================================
# Hyperparameters — edit these values to configure the model and training
# =============================================================================

# Model Architecture Lists for Grid Search
hidden_dims_GAT  = [64]       # Hidden dimension for GAT layers (as per paper)
out_dims         = [32, 64, 128]       # Output dimension of community-centric encoder
heads_list       = [5]        # Number of GAT attention heads (k=5 as per paper)
betas            = [0.44]     # eMRF trade-off parameter (as per paper)
mlp_hidden_dims  = [32, 64, 128]       # Hidden dimension for the edge classification MLP
nn_t_hidden_dims = [128]      # Hidden dimension for node classification MLP (as per paper)

# Loss weights
c1_list         = [1] # Group loss weight
c2_list         = [0.25, 0.5, 1] # Node loss weight
c3_list         = [0.25, 0.5, 1] # Transaction loss weight

# Training & CV
learning_rates  = [0.001]    # Adam optimizer learning rate
max_norm_clipping = [5.0]
epochs_model_selection = 100        # Epochs for Cross Validation
epochs_train    = 100         # Maximum number of training epochs
minibatches     = True       # Train with minibatches
batch_size      = 512        # Increased from 128 to stabilize gradients
num_neighbors   = [10, 10]    # max number of neighbours sampled for each node in the batch, set to -1 -1 to have no limits
patience = 100 # patience during model selection
print_every     = 1

# Class Imbalance
downsample      = True     # Down-sample majority class (legitimate nodes) in training set


# Split dataset
# trainset
date_time_1 = '2022-09-07 14:55:00'
# validation set
date_time_2 = '2022-09-08 16:12:00'
# test set

In [6]:
import pandas as pd
from torch_geometric.data import Data

# Convert Timestamp column to datetime (use downsampled edges_features_df)
ts = pd.to_datetime(edges_features_df['Timestamp'])

# Thresholds for splits
thresh_val = pd.to_datetime('2022-09-07 14:55:00')
thresh_test = pd.to_datetime('2022-09-08 16:12:00')

# Create edge masks based on chronological splits
train_edge_mask = torch.tensor((ts < thresh_val).values, dtype=torch.bool)
val_edge_mask = torch.tensor(((ts >= thresh_val) & (ts < thresh_test)).values, dtype=torch.bool)
test_edge_mask = torch.tensor((ts >= thresh_test).values, dtype=torch.bool)

print(f"Train edges: {train_edge_mask.sum().item()} ({train_edge_mask.sum().item()/len(ts)*100:.1f}%)")
print(f"Val edges: {val_edge_mask.sum().item()} ({val_edge_mask.sum().item()/len(ts)*100:.1f}%)")
print(f"Test edges: {test_edge_mask.sum().item()} ({test_edge_mask.sum().item()/len(ts)*100:.1f}%)")

# Create node masks for NeighborLoader input_nodes
# A node is in the training set if it is connected to at least one training edge
train_nodes_idx = edge_index[:, train_edge_mask].flatten().unique()
train_node_mask = torch.zeros(x.shape[0], dtype=torch.bool)
train_node_mask[train_nodes_idx] = True

val_nodes_idx = edge_index[:, val_edge_mask].flatten().unique()
val_node_mask = torch.zeros(x.shape[0], dtype=torch.bool)
val_node_mask[val_nodes_idx] = True

test_nodes_idx = edge_index[:, test_edge_mask].flatten().unique()
test_node_mask = torch.zeros(x.shape[0], dtype=torch.bool)
test_node_mask[test_nodes_idx] = True

# Node mask: Using all True, as GAGNN expects a probability label for all nodes
node_mask = torch.ones(x.shape[0], dtype=torch.bool)

# Build PyG Data object
data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y_trans=y_trans, y_node=y_node)
data.node_mask = node_mask
data.edge_train_mask = train_edge_mask
data.edge_val_mask = val_edge_mask
data.edge_test_mask = test_edge_mask
data.train_node_mask = train_node_mask
data.val_node_mask = val_node_mask
data.test_node_mask = test_node_mask


Train edges: 38189 (73.2%)
Val edges: 4556 (8.7%)
Test edges: 9435 (18.1%)


### Model selection

In [ ]:
import os
import json
import itertools
from model.gagnn import GAGNN
from model.loss import GAGNNLoss
from torch_geometric.loader import NeighborLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if minibatches:
    train_loader = NeighborLoader(
        data, num_neighbors=num_neighbors, batch_size=batch_size,
        input_nodes=data.train_node_mask, shuffle=True
    )
    
    val_loader = NeighborLoader(
        data, num_neighbors=num_neighbors, batch_size=batch_size,
        input_nodes=data.val_node_mask, shuffle=False
    )
else:
    d_dev = data.to(device)
    train_loader = [d_dev]
    val_loader = [d_dev]

print(f"Batches per epoch (Train): {len(train_loader)} | (Val): {len(val_loader)}")

keys = ['max_norm', 'hidden_dim_GAT', 'out_dim', 'heads', 'beta', 'mlp_hidden_dim', 'nn_t_hidden_dim', 'lr', 'c1', 'c2', 'c3']
combinations = list(itertools.product(
    max_norm_clipping, hidden_dims_GAT, out_dims, heads_list, betas, mlp_hidden_dims, nn_t_hidden_dims, learning_rates, c1_list, c2_list, c3_list
))

print(f"Total hyperparameter combinations: {len(combinations)}\n")

best_val_loss = float('inf')
best_params = None

os.makedirs("saved_models", exist_ok=True)

for idx, combo in enumerate(combinations):
    params = dict(zip(keys, combo))
    print(f"--- Experiment {idx+1}/{len(combinations)} ---")
    print(params)
    
    model = GAGNN(
        node_in_dim=x.shape[1],
        edge_feat_dim=edge_attr.shape[1],
        hidden_dim=params['hidden_dim_GAT'],
        out_dim=params['out_dim'],
        heads=params['heads'],
        beta=params['beta'],
        mlp_hidden_dim=params['mlp_hidden_dim'],
        nn_t_hidden_dim=params['nn_t_hidden_dim'],
        minibatches=minibatches
    ).to(device)
    
    criterion = GAGNNLoss(c1=params['c1'], c2=params['c2'], c3=params['c3'])
    optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'])
    
    epochs_no_improve = 0
    best_model_val_loss = float('inf')
    
    for epoch in range(epochs_model_selection):
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            p_node, p_trans, p_group, y_group = model(
                batch.x, batch.edge_index, batch.edge_attr, batch.edge_index.size(1),
                y_trans=batch.y_trans, trans_mask=batch.edge_train_mask
            )
            
            if p_group.numel() > 0:
                loss, _, _, _ = criterion(
                    p_node, batch.y_node.view(-1, 1),
                    p_trans, batch.y_trans,
                    p_group, y_group,
                    node_mask=batch.node_mask,
                    trans_mask=batch.edge_train_mask
                )
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=params['max_norm'])
                optimizer.step()
                epoch_loss += loss.item()
                n_batches += 1
                
        train_loss = epoch_loss / max(n_batches, 1)
        
        model.eval()
        val_loss_total = 0.0
        n_val_batches = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                p_node, p_trans, p_group, y_group = model(
                    batch.x, batch.edge_index, batch.edge_attr, batch.edge_index.size(1),
                    y_trans=batch.y_trans, trans_mask=batch.edge_val_mask
                )
                if p_group.numel() > 0:
                    val_loss, _, _, _ = criterion(
                        p_node, batch.y_node.view(-1, 1),
                        p_trans, batch.y_trans,
                        p_group, y_group,
                        node_mask=batch.node_mask,
                        trans_mask=batch.edge_val_mask
                    )
                    val_loss_total += val_loss.item()
                    n_val_batches += 1
                    
        avg_val_loss = val_loss_total / max(n_val_batches, 1)
        
        if avg_val_loss < best_model_val_loss:
            best_model_val_loss = avg_val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"patience triggered")
            print(f"Epoch {epoch+1:03d} | Train Loss: {train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
            break
            
        if (epoch + 1) % print_every == 0:
            print(f"Epoch {epoch+1:03d} | Train Loss: {train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")   
        
            
    print(f"Best Val Loss for this config: {best_model_val_loss:.4f}")
    if best_model_val_loss < best_val_loss:
        best_val_loss = best_model_val_loss
        best_params = params
        model.save("saved_models/gagnn_best_model_selection.pt")
        print(">>> New Best Parameters! Model saved.")

print("\n=========================================")
print(f"Overall Best Val Loss: {best_val_loss:.4f}")
print(f"Best Parameters: {best_params}")
print("=========================================")
with open("saved_models/best_params.json", "w") as f:
    json.dump(best_params, f, indent=4)

print("\nPlotting training history of the best model from model selection...")
best_selection_model = GAGNN.load_saved("saved_models/gagnn_best_model_selection.pt")
best_selection_model.plot_training_history()


Batches per epoch (Train): 22 | (Val): 5
Total hyperparameter combinations: 81

--- Experiment 1/81 ---
{'max_norm': 5.0, 'hidden_dim_GAT': 64, 'out_dim': 32, 'heads': 5, 'beta': 0.44, 'mlp_hidden_dim': 32, 'nn_t_hidden_dim': 128, 'lr': 0.001, 'c1': 1, 'c2': 0.25, 'c3': 0.25}
Epoch 001 | Train Loss: 10.2045 | Val Loss: 1.1172
Epoch 002 | Train Loss: 1.4998 | Val Loss: 0.8891
Epoch 003 | Train Loss: 2.9966 | Val Loss: 0.9596
Epoch 004 | Train Loss: 2.2965 | Val Loss: 0.8303
Epoch 005 | Train Loss: 0.9927 | Val Loss: 0.7391
Epoch 006 | Train Loss: 0.9153 | Val Loss: 1.0057
Epoch 007 | Train Loss: 1.3506 | Val Loss: 0.7235
Epoch 008 | Train Loss: 1.4829 | Val Loss: 0.8082
Epoch 009 | Train Loss: 0.8953 | Val Loss: 1.1448
Epoch 010 | Train Loss: 1.1323 | Val Loss: 0.7231
Epoch 011 | Train Loss: 0.9123 | Val Loss: 0.7518
Epoch 012 | Train Loss: 0.7503 | Val Loss: 0.7543
Epoch 013 | Train Loss: 0.8119 | Val Loss: 0.7399
Epoch 014 | Train Loss: 0.8426 | Val Loss: 0.7431
Epoch 015 | Train Loss

### Training

In [ ]:
import os
import json

print("\nLoading best hyperparameters for final training...")
with open("saved_models/best_params.json", "r") as f:
    best_params = json.load(f)

print("\nStarting final training on Train + Val datasets...")
# Combine masks
final_train_mask = data.edge_train_mask | data.edge_val_mask
final_train_node_mask = data.train_node_mask | data.val_node_mask

if minibatches:
    final_train_loader = NeighborLoader(
        data, num_neighbors=num_neighbors, batch_size=batch_size,
        input_nodes=final_train_node_mask, shuffle=True
    )
else:
    final_train_loader = [d_dev]

print(f"Batches per epoch (Final Train): {len(final_train_loader)}")

# Re-initialize the model with best parameters
final_model = GAGNN(
    node_in_dim=x.shape[1],
    edge_feat_dim=edge_attr.shape[1],
    hidden_dim=best_params['hidden_dim_GAT'],
    out_dim=best_params['out_dim'],
    heads=best_params['heads'],
    beta=best_params['beta'],
    mlp_hidden_dim=best_params['mlp_hidden_dim'],
    nn_t_hidden_dim=best_params['nn_t_hidden_dim'],
    minibatches=minibatches
).to(device)

final_criterion = GAGNNLoss(c1=best_params['c1'], c2=best_params['c2'], c3=best_params['c3'])
final_optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params['lr'])

os.makedirs("saved_models", exist_ok=True)

# No patience, just train for epochs_train
for epoch in range(epochs_train):
    final_model.train()
    epoch_loss = 0.0
    n_batches = 0
    
    for batch in final_train_loader:
        batch = batch.to(device)
        final_optimizer.zero_grad()
        
        p_node, p_trans, p_group, y_group = final_model(
            batch.x, batch.edge_index, batch.edge_attr, batch.edge_index.size(1),
            y_trans=batch.y_trans, trans_mask=final_train_mask
        )
        
        if p_group.numel() > 0:
            loss, _, _, _ = final_criterion(
                p_node, batch.y_node.view(-1, 1),
                p_trans, batch.y_trans,
                p_group, y_group,
                node_mask=batch.node_mask,
                trans_mask=final_train_mask
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(final_model.parameters(), max_norm=best_params['max_norm'])
            final_optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
            
    train_loss = epoch_loss / max(n_batches, 1)
    if (epoch + 1) % print_every == 0:
        print(f"Final Train Epoch {epoch+1:03d} | Train Loss: {train_loss:.4f}")
    
    # Save model every 5 epochs
    if (epoch + 1) % 5 == 0:
        final_model.save(f"saved_models/model_trained_{epoch+1}.pt")
        print(f"  -> Saved model checkpoint to saved_models/model_trained_{epoch+1}.pt")
        
final_model.save(f"saved_models/model_trained_final.pt")

print("\nPlotting training history of the final trained model...")
final_model.plot_training_history()


In [ ]:
from utils import Evaluator

print("\nLoading best model for final evaluation on Test set...")
eval_model = GAGNN.load_saved("saved_models/model_trained_final.pt").to(device)
eval_criterion = GAGNNLoss(c1=best_params['c1'], c2=best_params['c2'], c3=best_params['c3'])

if minibatches:
    test_loader = NeighborLoader(
        data, num_neighbors=num_neighbors, batch_size=batch_size,
        input_nodes=data.test_node_mask, shuffle=False
    )
    Evaluator.evaluation_report(eval_model, test_loader, eval_criterion, device)
else:
    Evaluator.evaluation_report(eval_model, [d_dev], eval_criterion, device)
